# Drug RAG — 벡터 DB v2 임베딩 (Colab)

**v1 대비 바뀐 점**
- 새 문서: `drug_permit`(식약처 허가정보), `dur_ingredient`(성분 단위 금기·주의) — **금기 검색은 `dur_ingredient`만 사용**
- `dur_interaction`(제품 쌍 금기, 약 83만 건)은 **보관용 별도 테이블**에만 넣고 검색에는 쓰지 않음 (기본값: 임베딩 안 함)
- 새 테이블 `drug_documents_v2` → v1 테이블(`drug_documents`)은 그대로 두고 비교 가능
- **이어서 실행**: 임베딩 결과를 S3에 파일 단위로 저장 → Colab 세션이 끊겨도 처음부터 다시 하지 않음
- 청크가 잘려도 **약품명/질환명 머리말**을 붙여 어떤 약의 내용인지 알 수 있게 함
- `doc_type`별 **부분 HNSW 인덱스** → 필터 검색 시 결과가 비는 문제 방지

**실행 전 체크리스트**
- [ ] 런타임 → 런타임 유형 변경 → **T4 GPU**
- [ ] 왼쪽 🔑 Secrets 등록: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `S3_BUCKET` (노트북 액세스 허용)

**흐름**
```
[1단계 · GPU] Silver(S3) → 청킹 → 임베딩 → S3 체크포인트 (gold/vector_v2/parts/)
[2단계 · CPU] Colab PostgreSQL + pgvector ← 체크포인트 적재 → 인덱스 → 검색 테스트 → pg_dump → S3
[로컬]       S3의 dump → Docker PostgreSQL에 복원 (맨 아래 명령어)
```

**예상 시간 (T4 기준, 대략)**: 1단계 20~40분, 2단계 20~30분. `INCLUDE_ARCHIVE=True`면 1단계에 1.5~2시간 추가.

⚠️ 세션이 끊기면: 런타임 다시 연결 → **0~4번 셀 실행 → 1단계 셀 다시 실행** (완료된 파일은 자동으로 건너뜀)

## 0. 설정

In [1]:
# Cell 0: 설정
RUN_ID = 'v2'                         # S3 체크포인트 폴더 이름. 같은 값이면 이어서 진행, 처음부터 다시 하려면 바꾸기
TABLE = 'drug_documents_v2'           # 검색용 테이블 (v1: drug_documents)
ARCHIVE_TABLE = 'dur_interaction_archive'  # 보관용 테이블 (검색에 쓰지 않음)

INCLUDE_ARCHIVE = False               # dur_interaction(제품 쌍 금기, 약 83만 건)까지 임베딩할지. 서비스 동작에는 필요 없음

# 허가정보(drug_permit) 범위
PERMIT_SUMMARY_SCOPE = 'active'       # 제품 요약(효능·용법): 'active' = 허가 유효 완제의약품 / 'all' = 전체
PERMIT_CAUTION_SCOPE = 'otc'          # 사용상주의사항 원문: 'otc' = 유효 완제 일반의약품만 / 'active' / 'none'

# 날짜 폴더 전체를 읽을 원천 (날짜별로 나눠 저장된 데이터). 나머지는 최신 날짜 폴더만 읽음
# 예) silver/refined/drugs: 2026-09-07에 48개 + 2026-09-08에 1개 → 최신만 읽으면 1개 파일만 들어감
ALL_DATES_GROUPS = {'drug_refined', 'symptom'}

BATCH_SIZE = 128                      # GPU 메모리 부족(OOM) 에러가 나면 64로 낮추기
CHECKPOINT_PREFIX = f'gold/vector_{RUN_ID}/parts'
print(f'RUN_ID={RUN_ID} | TABLE={TABLE} | 보관용 포함={INCLUDE_ARCHIVE}')

RUN_ID=v2 | TABLE=drug_documents_v2 | 보관용 포함=False


In [2]:
# Cell 1: GPU 확인
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} / VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU 없음 → 런타임 > 런타임 유형 변경 > T4 GPU (CPU로는 수십 배 느림)')

디바이스: cuda
GPU: Tesla T4 / VRAM 15.6 GB


In [3]:
# Cell 2: 패키지 설치
!pip install -q boto3 sentence-transformers psycopg2-binary pgvector langchain-text-splitters pyarrow tqdm
print('패키지 설치 완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.3 MB/s eta 0:00:00
패키지 설치 완료


In [4]:
# Cell 3: Secrets + S3 유틸
import os, io, re, json, time
import boto3
from botocore.exceptions import ClientError
from google.colab import userdata

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = userdata.get('AWS_DEFAULT_REGION')
S3_BUCKET = userdata.get('S3_BUCKET')
s3 = boto3.client('s3', region_name=os.environ['AWS_DEFAULT_REGION'])

DATE_RE = re.compile(r'/(\d{4}-\d{2}-\d{2})/')

def list_keys(prefix, suffix='.json'):
    keys = []
    for page in s3.get_paginator('list_objects_v2').paginate(Bucket=S3_BUCKET, Prefix=prefix):
        keys += [o['Key'] for o in page.get('Contents', []) if o['Key'].endswith(suffix)]
    return sorted(keys)

def latest_keys(prefix):
    # prefix 아래 날짜 폴더 중 가장 최신 날짜의 파일만 (ETL을 여러 번 돌려 날짜 폴더가 여러 개여도 안전)
    keys = list_keys(prefix)
    dates = sorted({m.group(1) for k in keys if (m := DATE_RE.search(k[len(prefix) - 1:]))})
    if not dates:
        return [], None
    return [k for k in keys if f'/{dates[-1]}/' in k], dates[-1]

def all_date_keys(prefix):
    # 모든 날짜 폴더의 파일 (최신 날짜 먼저 → 같은 문서는 최신 날짜 것이 우선)
    keys = list_keys(prefix)
    dated = sorted([(m.group(1), k) for k in keys if (m := DATE_RE.search(k[len(prefix) - 1:]))],
                   key=lambda x: (x[0], x[1]), reverse=True)
    dates = sorted({d for d, _ in dated})
    return [k for _, k in dated], (f'{dates[0]}~{dates[-1]}' if dates else None)

def source_keys(group, prefix):
    return all_date_keys(prefix) if group in ALL_DATES_GROUPS else latest_keys(prefix)

def part_name(group, key):
    # 체크포인트 파일명. 날짜 전체를 읽는 원천은 날짜가 달라도 파일명(batch_0000)이 같을 수 있어 날짜를 붙임
    base = key.rsplit('/', 1)[-1].replace('.json', '.parquet')
    return f'{DATE_RE.search(key).group(1)}_{base}' if group in ALL_DATES_GROUPS else base

def download_json(key):
    return json.loads(s3.get_object(Bucket=S3_BUCKET, Key=key)['Body'].read().decode('utf-8'))

def s3_exists(key):
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=key)
        return True
    except ClientError:
        return False

print(f'S3 버킷: {S3_BUCKET}')

S3 버킷: drug-rag-datalake-331145994962


In [5]:
# Cell 4: 임베딩 모델 로드 (v1과 같은 모델 → v1/v2 비교 가능)
from sentence_transformers import SentenceTransformer
MODEL_NAME = 'intfloat/multilingual-e5-large'
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 512
if device == 'cuda':
    model.half()                      # fp16 → T4에서 약 2배 빠름 (정규화 후 품질 차이 미미)
EMBED_DIM = model.get_sentence_embedding_dimension()
print(f'{MODEL_NAME} / 차원 {EMBED_DIM}')

def embed_passages(texts):
    # e5 모델은 문서에 'passage: ', 질문에 'query: ' 접두사가 필수
    return model.encode([f'passage: {t}' for t in texts], batch_size=BATCH_SIZE,
                        normalize_embeddings=True, convert_to_numpy=True,
                        show_progress_bar=False).astype('float32')

def embed_query(q):
    return model.encode(f'query: {q}', normalize_embeddings=True, convert_to_numpy=True).astype('float32')

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

intfloat/multilingual-e5-large / 차원 1024


/tmp/ipykernel_1080/547551594.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = model.get_sentence_embedding_dimension()


## 1단계 · 청킹 + 임베딩 (GPU)

### 벡터 DB v2에 들어가는 문서
| doc_type | 내용 | 원천 (S3, 최신 날짜 폴더만) | 청킹 |
|---|---|---|---|
| `drug_efficacy` / `drug_usage` / `drug_caution` / `drug_side_effect` | 일반의약품 효능·용법·주의사항·부작용 | `silver/refined/drugs/` | 512 / 256 / 384 / 384자 |
| `symptom_info` / `symptom_redflag` | 질환 설명 / 위험 증상 | `silver/refined/symptoms/` | 512자 / 나누지 않음 |
| `drug_permit` 🆕 | 허가 제품 요약(`section=summary`) + 사용상주의사항 원문(`section=caution`) | `silver/drugs/drug_permit/` | 512 / 384자 |
| `dur_ingredient` 🆕 | **성분 단위 금기·주의 (금기 검색은 이것만)** | `silver/dur_ingredient/{유형}/` | 나누지 않음 |
| `dur_interaction` (보관용) | 제품 쌍 금기 → **별도 테이블**, `INCLUDE_ARCHIVE=True`일 때만 | `silver/dur_interactions/{유형}/` | 나누지 않음 |

넣지 않는 것: `silver/drug_ingredients/`(제품-성분 구성), `silver/dur_ingredient/code_map/`(코드 변환표) → 그래프 전용

In [6]:
# Cell 5: 청킹 + 행(row) 생성
import re, json
from langchain_text_splitters import RecursiveCharacterTextSplitter

DUR_TYPES = ['combination_ban', 'pregnancy_ban', 'age_specific_ban', 'elderly_caution',
             'dose_caution', 'dosing_period_caution', 'effect_duplicate']

CHUNK_CONFIG = {
    'drug_efficacy':    (512, 64),
    'drug_usage':       (256, 32),
    'drug_caution':     (384, 64),
    'drug_side_effect': (384, 64),
    'symptom_info':     (512, 64),
    'drug_permit':      (512, 64),
    'drug_permit_caution': (384, 64),
}
NO_SPLIT = {'symptom_redflag', 'dur_ingredient', 'dur_interaction'}
_splitters = {}

def chunk_text(text, size, overlap):
    text = (text or '').strip()
    if not text:
        return []
    if len(text) <= size:
        return [text]
    if (size, overlap) not in _splitters:
        _splitters[(size, overlap)] = RecursiveCharacterTextSplitter(
            chunk_size=size, chunk_overlap=overlap,
            separators=['\n', ' | ', '. ', ', ', ' ', ''])
    out = []
    for c in _splitters[(size, overlap)].split_text(text):
        c = re.sub(r'^[\s|.,]+', '', c).rstrip(' |')   # 분할 기호가 청크 앞에 남는 것 정리
        if c:
            out.append(c)
    return out

def with_header(chunk, header):
    # 잘린 청크에도 '약품명: OO' 같은 머리말을 붙여 어떤 약/질환의 내용인지 알 수 있게 함
    if not header or chunk.startswith(header.split(':')[0]):
        return chunk
    return f'{header} | {chunk}'

def make_row(doc, meta, doc_type, idx, text, section='', suffix='', name=None):
    return {
        'doc_id':       f"{doc['doc_id']}{suffix}_c{idx}",
        'doc_type':     doc_type,
        'chunk_index':  idx,
        'section':      section or meta.get('section', '') or '',
        'text':         text,
        'name':         name if name is not None else (meta.get('name') or meta.get('item_name') or ''),
        'item_seq':     meta.get('item_seq', '') or '',
        'source_name':  meta.get('source_name', '') or '',
        'source_url':   meta.get('source_url', '') or '',
        'license':      meta.get('license', '') or '',
        'provider':     meta.get('provider', '') or '',
        'item_url':     meta.get('item_url') or meta.get('official_url') or '',
        'manufacturer': meta.get('manufacturer', '') or '',
        'image_url':    meta.get('image_url', '') or '',
        'dur_type':     meta.get('dur_type', '') or '',
        'level':        meta.get('level', '') or '',
        'ingredient_a': meta.get('ingredient_a_kor') or meta.get('ingredient_a_eng') or '',
        'ingredient_b': meta.get('ingredient_b_kor') or meta.get('ingredient_b_eng') or '',
        'is_otc':       meta.get('is_otc'),
        'is_active':    meta.get('is_active'),
        'metadata':     json.dumps(meta, ensure_ascii=False),
    }

def rows_refined(doc):
    # silver/refined/drugs, silver/refined/symptoms
    meta, doc_type = doc.get('metadata', {}), doc.get('doc_type', '')
    name = meta.get('name', '')
    header = (f'질환명: {name}' if doc_type.startswith('symptom') else f'약품명: {name}') if name else ''
    if doc_type in NO_SPLIT:
        chunks = chunk_text(doc.get('text', ''), 10**9, 0)
    else:
        size, overlap = CHUNK_CONFIG.get(doc_type, (512, 64))
        chunks = chunk_text(doc.get('text', ''), size, overlap)
    return [make_row(doc, meta, doc_type, i, with_header(c, header)) for i, c in enumerate(chunks)]

def rows_dur_ingredient(doc):
    meta = doc.get('metadata', {})
    a = meta.get('ingredient_a_kor') or meta.get('ingredient_a_eng') or ''
    b = meta.get('ingredient_b_kor') or meta.get('ingredient_b_eng') or ''
    text = (doc.get('text') or '').strip()
    return [make_row(doc, meta, 'dur_ingredient', 0, text, name=f'{a} + {b}' if b else a)] if text else []

def rows_permit(doc):
    meta = doc.get('metadata', {})
    active = bool(meta.get('is_active')) and bool(meta.get('is_finished_drug', True))
    otc = active and bool(meta.get('is_otc'))
    header = f"약품명: {meta.get('name', '')}"
    rows = []
    if PERMIT_SUMMARY_SCOPE == 'all' or active:
        for i, c in enumerate(chunk_text(doc.get('text', ''), *CHUNK_CONFIG['drug_permit'])):
            rows.append(make_row(doc, meta, 'drug_permit', i, with_header(c, header), section='summary'))
    caution = ((doc.get('sections') or {}).get('caution') or '').strip()
    want = {'none': False, 'otc': otc, 'active': active, 'all': True}[PERMIT_CAUTION_SCOPE]
    if caution and want:
        for i, c in enumerate(chunk_text(caution, *CHUNK_CONFIG['drug_permit_caution'])):
            rows.append(make_row(doc, meta, 'drug_permit', i, f'{header} | 주의사항: {c}',
                                 section='caution', suffix='_caution'))
    return rows

def rows_archive(doc):
    meta = doc.get('metadata', {})
    text = (doc.get('text') or '').strip()
    return [make_row(doc, meta, 'dur_interaction', 0, text)] if text else []

BUILDERS = {'refined': rows_refined, 'dur_ingredient': rows_dur_ingredient,
            'permit': rows_permit, 'archive': rows_archive}

SOURCES = [
    ('drug_refined',   'silver/refined/drugs/',      'refined'),
    ('symptom',        'silver/refined/symptoms/',   'refined'),
    ('drug_permit',    'silver/drugs/drug_permit/',  'permit'),
    *[(f'dur_ingredient_{t}', f'silver/dur_ingredient/{t}/', 'dur_ingredient') for t in DUR_TYPES],
]
ARCHIVE_SOURCES = [(f'dur_interaction_{t}', f'silver/dur_interactions/{t}/', 'archive') for t in DUR_TYPES]
print(f'검색용 원천 {len(SOURCES)}개 / 보관용 원천 {len(ARCHIVE_SOURCES)}개 (INCLUDE_ARCHIVE={INCLUDE_ARCHIVE})')

검색용 원천 10개 / 보관용 원천 7개 (INCLUDE_ARCHIVE=False)


In [7]:
# Cell 6: (선택) 미리보기 ─ 원천별 첫 파일로 청크가 어떻게 만들어지는지 확인 (GPU 사용 안 함)
for group, prefix, kind in SOURCES[:4]:
    keys, date = source_keys(group, prefix)
    if not keys:
        print(f'⚠️ {prefix}: 파일 없음'); continue
    items = download_json(keys[0]).get('items', [])
    rows = [r for d in items for r in BUILDERS[kind](d)]
    print(f'\n▶ {group} ({date}, 파일 {len(keys)}개) 첫 파일: 문서 {len(items)}건 → 청크 {len(rows)}개')
    for r in rows[:2]:
        print(f"   [{r['doc_type']}/{r['section'] or '-'}] {r['doc_id']}\n     {r['text'][:150]}")


▶ drug_refined (2026-09-07~2026-09-08, 파일 49개) 첫 파일: 문서 373건 → 청크 444개
   [drug_efficacy/efficacy] public_195700020_efficacy_c0
     약품명: 활명수 | 효능: 이 약은 식욕감퇴(식욕부진), 위부팽만감, 소화불량, 과식, 체함, 구역, 구토에 사용합니다.
   [drug_usage/usage] public_195700020_usage_c0
     약품명: 활명수 | 용법: 만 15세 이상 및 성인은 1회 1병(75 mL), 만 11세이상~만 15세미만은 1회 2/3병(50 mL), 만 8세 이상~만 11세 미만은 1회 1/2병(37.5 mL), 만 5세 이상~만 8세 미만은 1회 1/3병(25 mL), 만 3세

▶ symptom (2026-09-08~2026-09-08, 파일 5개) 첫 파일: 문서 5건 → 청크 8개
   [symptom_info/-] portal_SW_혈변 및 흑변(성인)_redflag_info_c0
     질환명: 혈변 및 흑변(성인)
   [symptom_redflag/-] portal_SW_혈변 및 흑변(성인)_redflag_redflag_c0
     질환명: 혈변 및 흑변(성인) | [요약문] '이것만은 꼭 기억하세요' • 혈변과 흑변은 소화관내부에서 출혈이 발생해 변에 혈액이 섞여 나오는 증상으로 혈변은 선홍색이나 적갈색, 흑변은 검고 반질거리는 변을 말합니다. • 혈변은 주로 하부위장관(소장, 대장, 직장) 출

▶ drug_permit (2026-10-07, 파일 428개) 첫 파일: 문서 100건 → 청크 223개
   [drug_permit/summary] permit_195700004_c0
     약품명: 대한포도당주사액(10%) | 구분: 전문의약품 | 주성분: 포도당 | 효능: 1. 고칼륨혈증, 순환허탈, 저혈당시의 에너지 보급: 2. 심질환(G,I,K요법) 그 외 수분, 에너지 보급: 3. 약물ㆍ독물 중독

In [8]:
# Cell 7: 1단계 실행 ─ 청킹 + 임베딩 + S3 체크포인트 저장 (이어서 실행 가능)
import numpy as np, pyarrow as pa, pyarrow.parquet as pq
from tqdm.auto import tqdm

COLUMNS = ['doc_id', 'doc_type', 'chunk_index', 'section', 'text', 'name', 'item_seq', 'source_name',
           'source_url', 'license', 'provider', 'item_url', 'manufacturer', 'image_url', 'dur_type',
           'level', 'ingredient_a', 'ingredient_b', 'is_otc', 'is_active', 'metadata']

def to_parquet_bytes(rows, emb):
    data = {c: [r[c] for r in rows] for c in COLUMNS}
    table = pa.table(data)
    vec = pa.FixedSizeListArray.from_arrays(pa.array(emb.reshape(-1), type=pa.float32()), EMBED_DIM) \
        if len(rows) else pa.array([], type=pa.list_(pa.float32(), EMBED_DIM))
    table = table.append_column('embedding', vec)
    buf = io.BytesIO()
    pq.write_table(table, buf, compression='zstd')
    return buf.getvalue()

def process_source(group, prefix, kind, stats):
    keys, date = source_keys(group, prefix)
    if not keys:
        print(f'⚠️ 스킵 (파일 없음): {prefix}'); return
    done = skipped = 0
    for key in tqdm(keys, desc=f'{group} ({date})', leave=False):
        part_key = f"{CHECKPOINT_PREFIX}/{group}/{part_name(group, key)}"
        if s3_exists(part_key):
            skipped += 1; continue
        rows, seen = [], set()
        for d in download_json(key).get('items', []):
            for r in BUILDERS[kind](d):
                if r['doc_id'] not in seen:
                    seen.add(r['doc_id']); rows.append(r)
        emb = embed_passages([r['text'] for r in rows]) if rows else np.zeros((0, EMBED_DIM), 'float32')
        s3.put_object(Bucket=S3_BUCKET, Key=part_key, Body=to_parquet_bytes(rows, emb))   # 빈 파일도 저장 → 완료 표시
        done += 1
        stats[group] = stats.get(group, 0) + len(rows)
    print(f'✅ {group:32} 새로 처리 {done:4}개 파일 / 이미 완료 {skipped:4}개 / 이번에 만든 청크 {stats.get(group, 0):,}개')

t0, stats = time.time(), {}
for src in SOURCES + (ARCHIVE_SOURCES if INCLUDE_ARCHIVE else []):
    process_source(*src, stats)
print(f'\n1단계 완료: 이번 실행에서 청크 {sum(stats.values()):,}개 임베딩 / {(time.time() - t0) / 60:.1f}분')

drug_refined (2026-09-07~2026-09-08):   0%|          | 0/49 [00:00<?, ?it/s]

✅ drug_refined                     새로 처리   49개 파일 / 이미 완료    0개 / 이번에 만든 청크 32,334개


symptom (2026-09-08~2026-09-08):   0%|          | 0/5 [00:00<?, ?it/s]

✅ symptom                          새로 처리    5개 파일 / 이미 완료    0개 / 이번에 만든 청크 2,132개


drug_permit (2026-10-07):   0%|          | 0/428 [00:00<?, ?it/s]

...

✅ dur_ingredient_dose_caution      새로 처리    0개 파일 / 이미 완료    2개 / 이번에 만든 청크 0개


dur_ingredient_dosing_period_caution (2026-10-08):   0%|          | 0/1 [00:00<?, ?it/s]

✅ dur_ingredient_dosing_period_caution 새로 처리    0개 파일 / 이미 완료    1개 / 이번에 만든 청크 0개


dur_ingredient_effect_duplicate (2026-10-08):   0%|          | 0/1 [00:00<?, ?it/s]

✅ dur_ingredient_effect_duplicate  새로 처리    0개 파일 / 이미 완료    1개 / 이번에 만든 청크 0개

1단계 완료: 이번 실행에서 청크 34,466개 임베딩 / 4.3분


## 2단계 · PostgreSQL 적재 + 인덱스 + 검색 테스트 (GPU 없어도 됨)

1단계가 끝났다면 GPU 없이 진행해도 됩니다. 세션이 끊긴 뒤 2단계만 다시 하려면 **0, 2, 3, 4번 셀 → 8번부터** 실행하세요. (4번은 검색 테스트용 모델 로드)

In [9]:
# Cell 8: PostgreSQL 15 + pgvector 설치
# 로컬 Docker(pgvector/pgvector:pg15)와 메이저 버전을 맞춰야 덤프가 그대로 복원됨
# (Colab 기본 PostgreSQL은 16 이상 → 로컬 15에서 'unsupported version (1.15) in file header' 에러)
import subprocess
PG_MAJOR = '15'   # 로컬 PostgreSQL 메이저 버전과 반드시 같게

!apt-get -qq update > /dev/null
!apt-get -qq install -y postgresql-common > /dev/null
!yes | /usr/share/postgresql-common/pgdg/apt.postgresql.org.sh > /dev/null 2>&1
!apt-get -qq install -y postgresql-{PG_MAJOR} postgresql-{PG_MAJOR}-pgvector > /dev/null

installed = sorted(os.listdir('/usr/lib/postgresql'), key=int)
print('설치된 PostgreSQL:', installed)
assert installed == [PG_MAJOR], f'{PG_MAJOR} 외 다른 버전이 설치됨 → 덤프 형식이 달라질 수 있음'
!ls /usr/share/postgresql/{PG_MAJOR}/extension/vector.control && echo '✅ pgvector 설치 완료'

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
설치된 PostgreSQL: ['15']
/usr/share/postgresql/15/extension/vector.control
✅ pgvector 설치 완료


In [10]:
# Cell 9: PostgreSQL 시작 + DB/테이블 생성
import psycopg2
from psycopg2.extras import execute_values

!service postgresql start
time.sleep(3)
!sudo -u postgres psql -q -c "ALTER USER postgres PASSWORD 'drugrag123';"
!sudo -u postgres psql -q -tc "SELECT 1 FROM pg_database WHERE datname='drug_rag'" | grep -q 1 || sudo -u postgres psql -q -c "CREATE DATABASE drug_rag;"
!sudo -u postgres psql -q -d drug_rag -c "CREATE EXTENSION IF NOT EXISTS vector;"

conn = psycopg2.connect(host='localhost', port=5432, dbname='drug_rag', user='postgres', password='drugrag123')
from pgvector.psycopg2 import register_vector
register_vector(conn)

DDL = '''
CREATE TABLE IF NOT EXISTS {t} (
    id            SERIAL PRIMARY KEY,
    doc_id        TEXT UNIQUE NOT NULL,
    doc_type      TEXT NOT NULL,
    chunk_index   INTEGER DEFAULT 0,
    section       TEXT,
    text          TEXT NOT NULL,
    embedding     vector({dim}),
    name          TEXT,
    item_seq      TEXT,
    source_name   TEXT,
    source_url    TEXT,
    license       TEXT,
    provider      TEXT,
    item_url      TEXT,
    manufacturer  TEXT,
    image_url     TEXT,
    dur_type      TEXT,
    level         TEXT,
    ingredient_a  TEXT,
    ingredient_b  TEXT,
    is_otc        BOOLEAN,
    is_active     BOOLEAN,
    data_version  TEXT DEFAULT '{ver}',
    metadata      JSONB,
    created_at    TIMESTAMP DEFAULT NOW()
);'''
with conn.cursor() as cur:
    cur.execute(DDL.format(t=TABLE, dim=EMBED_DIM, ver=RUN_ID))
    if INCLUDE_ARCHIVE:
        cur.execute(DDL.format(t=ARCHIVE_TABLE, dim=EMBED_DIM, ver=RUN_ID))
conn.commit()
print(f'✅ 테이블 준비: {TABLE}' + (f', {ARCHIVE_TABLE}' if INCLUDE_ARCHIVE else ''))

[ OK ]
✅ 테이블 준비: drug_documents_v2


In [11]:
# Cell 10: S3 체크포인트 → PostgreSQL 적재
INSERT_COLS = COLUMNS + ['embedding']

def load_group(group, table):
    keys = list_keys(f'{CHECKPOINT_PREFIX}/{group}/', suffix='.parquet')
    if group in ALL_DATES_GROUPS:
        dated = [k for k in keys if re.match(r'\d{4}-\d{2}-\d{2}_', k.rsplit('/', 1)[-1])]
        if len(dated) < len(keys):
            print(f'   ↳ {group}: 날짜 없는 예전 체크포인트 {len(keys) - len(dated)}개는 건너뜀')
        keys = sorted(dated, reverse=True)   # 최신 날짜 먼저 적재 → 같은 doc_id는 최신 것이 남음
    n = 0
    for key in tqdm(keys, desc=group, leave=False):
        t = pq.read_table(io.BytesIO(s3.get_object(Bucket=S3_BUCKET, Key=key)['Body'].read()))
        if t.num_rows == 0:
            continue
        cols = t.to_pydict()
        emb = np.asarray(t.column('embedding').combine_chunks().flatten(), dtype='float32').reshape(-1, EMBED_DIM)
        values = [tuple(cols[c][i] for c in COLUMNS) + (emb[i],) for i in range(t.num_rows)]
        with conn.cursor() as cur:
            execute_values(cur, f"INSERT INTO {table} ({', '.join(INSERT_COLS)}) VALUES %s "
                                f"ON CONFLICT (doc_id) DO NOTHING", values, page_size=500)
        conn.commit()
        n += t.num_rows
    return n

t0 = time.time()
for group, _, _ in SOURCES:
    print(f'  {group:32} {load_group(group, TABLE):>8,}행')
if INCLUDE_ARCHIVE:
    for group, _, _ in ARCHIVE_SOURCES:
        print(f'  {group:32} {load_group(group, ARCHIVE_TABLE):>8,}행 (보관용)')
print(f'적재 완료 / {(time.time() - t0) / 60:.1f}분')

   ↳ drug_refined: 날짜 없는 예전 체크포인트 1개는 건너뜀


drug_refined:   0%|          | 0/49 [00:00<?, ?it/s]

  drug_refined                       32,334행
   ↳ symptom: 날짜 없는 예전 체크포인트 5개는 건너뜀


symptom:   0%|          | 0/5 [00:00<?, ?it/s]

  symptom                             2,132행


...

  dur_ingredient_dose_caution           637행


dur_ingredient_dosing_period_caution:   0%|          | 0/1 [00:00<?, ?it/s]

  dur_ingredient_dosing_period_caution       95행


dur_ingredient_effect_duplicate:   0%|          | 0/1 [00:00<?, ?it/s]

  dur_ingredient_effect_duplicate       375행
적재 완료 / 12.2분


In [12]:
# Cell 11: 적재 결과 확인
with conn.cursor() as cur:
    cur.execute(f'''SELECT doc_type, COALESCE(NULLIF(section, ''), '-') AS section,
                           COUNT(*) AS chunks, COUNT(DISTINCT regexp_replace(doc_id, '(_caution)?_c[0-9]+$', '')) AS docs
                    FROM {TABLE} GROUP BY 1, 2 ORDER BY 1, 2''')
    rows = cur.fetchall()
print(f"  {'doc_type':20} {'section':8} {'청크':>9} {'원문서':>9}")
for dt, sec, chunks, docs in rows:
    print(f'  {dt:20} {sec:8} {chunks:>9,} {docs:>9,}')
print(f"  {'합계':28} {sum(r[2] for r in rows):>9,}")
print('''
참고 (원문서 기준 예상치)
  dur_ingredient 4,639 / drug_permit 요약: 허가 유효 완제의약품 수 / 주의사항: 일반의약품 약 8,500개
  drug_efficacy·usage·caution·side_effect, symptom_*: v1과 같은 원천''')

  doc_type             section         청크       원문서
  drug_caution         caution     13,756     8,404
  drug_efficacy        efficacy     8,350     8,350
  drug_permit          caution     60,273     8,598
  drug_permit          summary    103,252    35,195
  drug_side_effect     side_effect     1,580     1,179
  drug_usage           usage        8,648     8,368
  dur_ingredient       -            4,639     4,639
  symptom_info         -            1,811       397
  symptom_redflag      -              321       321
  합계                             202,630

참고 (원문서 기준 예상치)
  dur_ingredient 4,639 / drug_permit 요약: 허가 유효 완제의약품 수 / 주의사항: 일반의약품 약 8,500개
  drug_efficacy·usage·caution·side_effect, symptom_*: v1과 같은 원천


In [13]:
# Cell 12: 벡터 인덱스 ─ doc_type별 부분 HNSW 인덱스
# 검색은 항상 doc_type 필터와 함께 하므로, 종류별 인덱스를 따로 만들면
# '필터를 걸었더니 결과가 비는' 문제(근사 검색 후 필터링)가 생기지 않습니다.
t0 = time.time()
with conn.cursor() as cur:
    cur.execute("SET maintenance_work_mem = '2GB'")
    cur.execute("SET max_parallel_maintenance_workers = 2")
    for col in ['doc_type', 'dur_type', 'item_seq']:
        cur.execute(f'CREATE INDEX IF NOT EXISTS {TABLE}_{col}_idx ON {TABLE} ({col})')
    cur.execute(f'SELECT DISTINCT doc_type FROM {TABLE}')
    for (dt,) in cur.fetchall():
        cur.execute(f'''CREATE INDEX IF NOT EXISTS {TABLE}_hnsw_{dt} ON {TABLE}
                        USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 64)
                        WHERE doc_type = '{dt}' ''')
        conn.commit()
        print(f'  ✅ HNSW 인덱스: {dt}')
print(f'인덱스 생성 완료 / {(time.time() - t0) / 60:.1f}분  (보관용 테이블은 검색하지 않으므로 벡터 인덱스 없음)')

  ✅ HNSW 인덱스: drug_caution
  ✅ HNSW 인덱스: drug_efficacy
  ✅ HNSW 인덱스: drug_permit
  ✅ HNSW 인덱스: drug_side_effect
  ✅ HNSW 인덱스: drug_usage
  ✅ HNSW 인덱스: dur_ingredient
  ✅ HNSW 인덱스: symptom_info
  ✅ HNSW 인덱스: symptom_redflag
인덱스 생성 완료 / 2.4분  (보관용 테이블은 검색하지 않으므로 벡터 인덱스 없음)


In [14]:
# Cell 13: 검색 함수 + 테스트
def search(query, doc_type, top_k=5, dur_type=None, otc_only=False, dedup=True):
    # doc_type은 필수 (부분 인덱스 사용). dur_type / otc_only는 추가 필터
    q = embed_query(query)
    where, params = ['doc_type = %s'], [doc_type]
    if dur_type:
        where.append('dur_type = %s'); params.append(dur_type)
    if otc_only:
        where.append('is_otc AND is_active')
    sql = f'''SELECT name, section, text, source_name, metadata, 1 - (embedding <=> %s) AS score
              FROM {TABLE} WHERE {' AND '.join(where)}
              ORDER BY embedding <=> %s LIMIT %s'''
    with conn.cursor() as cur:
        cur.execute('SET hnsw.ef_search = 100')
        try:
            cur.execute('SET hnsw.iterative_scan = relaxed_order')   # pgvector 0.8+: 추가 필터가 있어도 결과 수 보장
        except Exception:
            conn.rollback()
        cur.execute(sql, [q, *params, q, top_k * 5])
        rows = cur.fetchall()
    out, seen = [], set()
    for name, section, text, source, meta, score in rows:
        key = (meta or {}).get('parent_doc_id') or (meta or {}).get('item_seq') or name
        if dedup and key in seen:
            continue
        seen.add(key)
        out.append((score, name, section, text))
        if len(out) >= top_k:
            break
    return out

def show(title, results):
    print(f'\n=== {title} ===')
    for score, name, section, text in results:
        print(f'[{score:.3f}] {name}' + (f' ({section})' if section else ''))
        print(f'    {text[:110]}')

show('두통에 먹는 약 (drug_efficacy)', search('두통에 먹는 약', 'drug_efficacy'))
show('이트라코나졸과 같이 먹으면 안 되는 약 (dur_ingredient)',
     search('이트라코나졸이랑 같이 먹으면 안 되는 약', 'dur_ingredient', dedup=False))
show('임산부가 먹으면 안 되는 진통제 (임부금기만)',
     search('임산부 진통제 금기', 'dur_ingredient', dur_type='pregnancy_ban', dedup=False))
show('어린이 해열제 (일반의약품 허가정보)', search('어린이 해열제 시럽', 'drug_permit', otc_only=True))
show('타이레놀 주의사항 원문 (drug_permit caution)', search('타이레놀 간 손상 음주', 'drug_permit', otc_only=True))
show('즉시 병원 가야 하는 증상 (symptom_redflag)', search('즉시 병원 가야 하는 증상', 'symptom_redflag'))


=== 두통에 먹는 약 (drug_efficacy) ===
[0.877] 졸리민정 (efficacy)
    약품명: 졸리민정 | 효능: 두통, 동통 및 불면증을 수반하는 통증, 발열의 일시적 완화
[0.876] 뇌선 (efficacy)
    약품명: 뇌선 | 효능: 이 약은 두통, 치통, 발치후 동통, 인후통, 귀의 통증, 관절통, 신경통, 요통, 근육통, 견통, 타박통, 골절통, 염좌통, 월경통(생리통), 외상통의 진통, 오한, 발열
[0.875] 게보린브이정(아세트아미노펜) (efficacy)
    약품명: 게보린브이정(아세트아미노펜) | 효능: 이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에 사용합니다
[0.874] 디큐펜맥스정500밀리그램(아세트아미노펜) (efficacy)
    약품명: 디큐펜맥스정500밀리그램(아세트아미노펜) | 효능: 이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에
[0.874] 내린다시럽(아세트아미노펜) (efficacy)
    약품명: 내린다시럽(아세트아미노펜) | 효능: 이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에 사용합니다.

=== 이트라코나졸과 같이 먹으면 안 되는 약 (dur_ingredient) ===
[0.885] 이트라코나졸 + 베네토클락스
    [병용금기] 성분: 이트라코나졸 + 베네토클락스 | 사유: 병용 시 이 약의 노출이 증가하여 TLS를 포함한 독성 위험 증가 가능 | 비고: 만성 림프구성 백혈병 환자에서 치료 시작 시와 용량 증량
[0.882] 피네레논 + 이트라코나졸
    [병용금기] 성분: 피네레논 + 이트라코나졸 | 사유: Finerenone의 혈중농도 증가, 치료 효과 및 이상반응 연장이 일어날 수 있음. Itrac

In [15]:
# Cell 14: pg_dump → S3 (세션 종료 전 필수!)  ─ v2 테이블만 덤프 → 로컬 v1 테이블은 건드리지 않음
from datetime import datetime
dump_path = f'/tmp/drug_rag_{RUN_ID}.dump'
dump_key = f"gold/pgvector_dump/{datetime.now().strftime('%Y-%m-%d')}/drug_rag_{RUN_ID}.dump"
tables = [TABLE] + ([ARCHIVE_TABLE] if INCLUDE_ARCHIVE else [])
cmd = ['sudo', '-u', 'postgres', f'/usr/lib/postgresql/{PG_MAJOR}/bin/pg_dump', '-Fc', '-d', 'drug_rag', '-f', dump_path]
for t in tables:
    cmd += ['-t', t]
r = subprocess.run(cmd, capture_output=True, text=True)
if r.returncode == 0:
    print(f'dump 크기: {os.path.getsize(dump_path) / 1024 / 1024:.1f} MB')
    s3.upload_file(dump_path, S3_BUCKET, dump_key)
    print(f'✅ S3 저장: s3://{S3_BUCKET}/{dump_key}')
else:
    print(f'오류: {r.stderr}')

dump 크기: 946.6 MB
✅ S3 저장: s3://drug-rag-datalake-331145994962/gold/pgvector_dump/2026-10-08/drug_rag_v2.dump


## 로컬 복원 (Windows CMD)

v2 테이블만 추가로 복원합니다. **기존 v1 테이블(`drug_documents`)은 그대로** 남습니다.

```cmd
:: 1) dump 다운로드 (날짜는 14번 셀 출력 참고)
aws s3 cp s3://drug-rag-datalake-331145994962/gold/pgvector_dump/YYYY-MM-DD/drug_rag_v2.dump .\drug_rag_v2.dump

:: 2) 로컬 pgvector 버전 확인 → 0.5.0 이상이어야 HNSW 인덱스 사용 가능 (0.8.0 이상 권장)
docker exec -it drug_rag_postgres psql -U airflow -d drug_rag -c "SELECT extversion FROM pg_extension WHERE extname='vector';"

:: 3) 기존 v2 테이블이 있으면 삭제 후 복원 (소유자는 airflow로 바뀌도록 --no-owner)
docker exec -it drug_rag_postgres psql -U airflow -d drug_rag -c "DROP TABLE IF EXISTS drug_documents_v2, dur_interaction_archive;"
docker cp .\drug_rag_v2.dump drug_rag_postgres:/tmp/
docker exec -it drug_rag_postgres pg_restore -U airflow -d drug_rag --no-owner --no-privileges /tmp/drug_rag_v2.dump

:: 4) 확인
docker exec -it drug_rag_postgres psql -U airflow -d drug_rag -c "SELECT doc_type, section, COUNT(*) FROM drug_documents_v2 GROUP BY 1, 2 ORDER BY 1, 2;"
```

## AI 파트 검색 예시 (v2)

```python
import psycopg2, numpy as np
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/multilingual-e5-large')
conn = psycopg2.connect(host='localhost', port=5432, dbname='drug_rag', user='airflow', password='...')
register_vector(conn)

def search(query, doc_type, top_k=5, **filters):
    q = model.encode(f'query: {query}', normalize_embeddings=True)       # 'query: ' 접두사 필수
    where = ['doc_type = %s'] + [f'{k} = %s' for k in filters]           # doc_type은 항상 지정 (부분 인덱스)
    with conn.cursor() as cur:
        cur.execute('SET hnsw.ef_search = 100')
        cur.execute(f'''SELECT name, text, source_name, source_url, metadata, 1 - (embedding <=> %s) AS score
                        FROM drug_documents_v2 WHERE {' AND '.join(where)}
                        ORDER BY embedding <=> %s LIMIT %s''',
                    [q, doc_type, *filters.values(), q, top_k])
        return cur.fetchall()

search('이트라코나졸 병용금기', 'dur_ingredient')                          # 금기 근거는 dur_ingredient만
search('임산부 진통제', 'dur_ingredient', dur_type='pregnancy_ban')
search('어린이 해열제', 'drug_permit', is_otc=True, is_active=True)
```

> "먹어도 되는가" 판정은 벡터 검색이 아니라 그래프(`graph_tools.assess()`)로 합니다. 벡터 DB는 판정 근거를 설명하는 데 씁니다.